In [1]:
from google.colab import files
uploaded = files.upload()

Saving Dataset .csv to Dataset .csv


In [2]:
import pandas as pd
df=pd.read_csv("Dataset .csv")
df.head()

,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,Botswana Pula(P),Yes,No,No,No,3,4.8,Dark Green,Excellent,314
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,...,Botswana Pula(P),Yes,No,No,No,3,4.5,Dark Green,Excellent,591
2,6300002,Heat - Edsa Shangri-La,162,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Ma...",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",...,Botswana Pula(P),Yes,No,No,No,4,4.4,Green,Very Good,270
3,6318506,Ooma,162,Mandaluyong City,"Third Floor, Mega Fashion Hall, SM Megamall, O...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.056475,14.585318,"Japanese, Sushi",...,Botswana Pula(P),No,No,No,No,4,4.9,Dark Green,Excellent,365
4,6314302,Sambo Kojin,162,Mandaluyong City,"Third Floor, Mega Atrium, SM Megamall, Ortigas...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.057508,14.584450,"Japanese, Korean",...,Botswana Pula(P),Yes,No,No,No,4,4.8,Dark Green,Excellent,229


In [3]:
df = df[['Restaurant Name', 'Cuisines', 'Price range']]
df.head()

,Restaurant Name,Cuisines,Price range
0,Le Petit Souffle,"French, Japanese, Desserts",3
1,Izakaya Kikufuji,Japanese,3
2,Heat - Edsa Shangri-La,"Seafood, Asian, Filipino, Indian",4
3,Ooma,"Japanese, Sushi",4
4,Sambo Kojin,"Japanese, Korean",4


In [4]:
df.isnull().sum()

,0
Restaurant Name,0
Cuisines,9
Price range,0


In [6]:
df.dropna(inplace=True)

In [7]:
df.isnull().sum()

,0
Restaurant Name,0
Cuisines,0
Price range,0


In [8]:
df['Features'] = df['Cuisines'] + ' ' + df['Price range'].astype(str)

In [9]:
df[['Restaurant Name', 'Features']].head()

,Restaurant Name,Features
0,Le Petit Souffle,"French, Japanese, Desserts 3"
1,Izakaya Kikufuji,Japanese 3
2,Heat - Edsa Shangri-La,"Seafood, Asian, Filipino, Indian 4"
3,Ooma,"Japanese, Sushi 4"
4,Sambo Kojin,"Japanese, Korean 4"


In [10]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(stop_words='english')

matrix = cv.fit_transform(df['Features'])

In [11]:
matrix.shape

(9542, 148)

In [12]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(matrix)

In [13]:
similarity.shape

(9542, 9542)

In [14]:
def recommend_restaurant(name):

    if name not in df['Restaurant Name'].values:
        return "Restaurant not found"

    idx = df[df['Restaurant Name'] == name].index[0]

    scores = list(enumerate(similarity[idx]))

    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    recommendations = []

    for i in scores[1:6]:
        recommendations.append(df.iloc[i[0]]['Restaurant Name'])

    return recommendations

In [15]:
df['Restaurant Name'].head(20)

,Restaurant Name
0,Le Petit Souffle
1,Izakaya Kikufuji
2,Heat - Edsa Shangri-La
3,Ooma
4,Sambo Kojin
5,Din Tai Fung
6,Buffet 101
7,Vikings
8,Spiral - Sofitel Philippine Plaza Manila
9,Locavore


In [16]:
recommend_restaurant("Le Petit Souffle")

['Tokyo Mon Amour',
 'Izakaya Kikufuji',
 'Sushi Loko',
 'New Koto',
 'Sushi Leblon']

In [17]:
def recommend_by_preference(cuisine, price):

    results = df[
        (df['Cuisines'].str.contains(cuisine, case=False, na=False)) &
        (df['Price range'] == price)
    ]

    return results[['Restaurant Name', 'Cuisines', 'Price range']].head(10)

In [18]:
recommend_by_preference("Japanese", 3)

,Restaurant Name,Cuisines,Price range
0,Le Petit Souffle,"French, Japanese, Desserts",3
1,Izakaya Kikufuji,Japanese,3
27,Sushi Loko,Japanese,3
97,Mikata Japanese Steakhouse,"Japanese, Steak, Sushi",3
98,Shogun Japanese Steak House,"Japanese, Steak, Sushi",3
193,Fuji Japanese Steak House,"Japanese, Steak",3
195,Samurai Japanese Cuisine & Sushi Bar,"Japanese, Steak, Sushi",3
196,Wasabi Sushi and Thai,"Japanese, Sushi, Thai",3
215,Fuji Japanese Steakhouse,"Japanese, Sushi",3
247,Osaka,"Japanese, Sushi",3


In [19]:
recommend_by_preference("North Indian", 2)

,Restaurant Name,Cuisines,Price range
625,Rangrezz Restaurant,"North Indian, Mughlai",2
626,Time2Eat - Mama Chicken,North Indian,2
643,Thaaliwala,"North Indian, Fast Food",2
656,Kabir Restaurant,"North Indian, Chinese, Continental, Desserts, ...",2
683,Sagar Ratna,"South Indian, North Indian, Chinese, Continental",2
685,Makhan Fish and Chicken Corner,"North Indian, Chinese",2
694,Sakhis Watz Kukin,"North Indian, Chinese",2
696,Shudh Restaurant,"North Indian, South Indian, Chinese, Fast Food",2
697,Bade Bhai Ka Brothers' Dhaba,"North Indian, Fast Food",2
698,Bharawan Da Dhaba,"North Indian, Fast Food",2
